In [10]:
import logging
import sys
import os
from glob import glob

import torch
from torch.utils.tensorboard import SummaryWriter
from torch.optim import Adam

import monai
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd
from monai.data import Dataset, DataLoader, list_data_collate
from monai.networks.nets import UNet
from monai.losses import DiceLoss

# Environment Config

In [ ]:
monai.config.print_config()
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

MONAI version: 1.6.0
Numpy version: 2.5.1
Pytorch version: 2.12.1+cu130
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: eccefc57550b111ed781d82249dfe77872a0e918
MONAI __file__: /homes/<username>/.conda/envs/project/lib/python3.14/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: 5.4.6
Nibabel version: 5.4.2
scikit-image version: NOT INSTALLED or UNKNOWN VERSION.
scipy version: NOT INSTALLED or UNKNOWN VERSION.
Pillow version: 12.3.0
Tensorboard version: 2.21.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.27.1+cu130
tqdm version: NOT INSTALLED or UNKNOWN VERSION.
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.2.2
pandas version: NOT INSTALLED or UNKNOWN VERSION.
einops version: NOT INSTALLED or UNKNOWN VERSION.
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd vers

# Data Preparation

In [12]:
train_dir = '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train'

images = sorted(glob(os.path.join(train_dir, 'images/*.png')))
masks = sorted(glob(os.path.join(train_dir, 'masks/*.png')))

train_files = [ {'img': img, 'mask': mask} for img, mask in zip(images, masks) ]
print(train_files[:5])

[{'img': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/images/000000.png', 'mask': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/masks/000000.png'}, {'img': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/images/000001.png', 'mask': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/masks/000001.png'}, {'img': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/images/000002.png', 'mask': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/masks/000002.png'}, {'img': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/images/000003.png', 'mask': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/masks/000003.png'}, {'img': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/images/000004.png', 'mask': '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train/masks/000004.png'}]


In [13]:
train_transforms = Compose(
    [
        LoadImaged(keys=['img', 'mask']),
        EnsureChannelFirstd(keys=['img', 'mask']),
    ]
)

# Data Load

In [15]:
train_dataset = Dataset(
    data=train_files,
    transform=train_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate,
)

# Parameter Config

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [8]:
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(device)

In [9]:
loss_fun = DiceLoss(
    sigmoid=True
)

In [10]:
optimizer = Adam(model.parameters(), lr=1e-3)

In [11]:
writer = SummaryWriter()

In [12]:
N_EPOCHES = 200

# Training

In [13]:
for epoch in range(N_EPOCHES):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{N_EPOCHES}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch in train_loader:
        step += 1
        imgs, masks = batch['img'].to(device), batch['mask'].to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = loss_fun(pred, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_dataset) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)
    epoch_loss /= step
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

writer.close()

----------
epoch 1/200


Exception ignored while calling weakref callback <function _WeakValueDictionary.__init__.<locals>.KeyedRef.remove at 0x7f13407305c0>:
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 91, in remove
KeyboardInterrupt: 


KeyboardInterrupt: 